In [1]:
from datetime import datetime
import pandas as pd
import numpy as np
import os 

## import helper 

In [2]:
platformID = 'FBE'

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from functions import execute_sql_query
import test_functions

from config import gam_info

In [4]:
# week 
week_cols = ['w/c']
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')

# social media accounts
channel_cols=['Channel ID']
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')

# socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Year'] == gam_info['file_timeinfo']]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['PlatformID'] == platformID]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Status'] == 'active']
socialmedia_accounts['Channel ID'] = socialmedia_accounts['Channel ID'].dropna().apply(lambda x: str(int(x)))
socialmedia_accounts['Channel ID'] = platformID + socialmedia_accounts['Channel ID']
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)

### RUN TESTS
test_functions.test_lookup_files(week_tester, ['w/c'], [f"{platformID}_1engage_3", f"{platformID}_1engage_4", f"{platformID}_1engage_5"], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [f"{platformID}_1engage_6", f"{platformID}_1engage_7", f"{platformID}_1engage_8"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")


✅ Test FBE_1engage_3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1engage_4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1engage_5 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_1engage_6 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1engage_7 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1engage_8 passed: No missing values in lookup.
...updating logbook...



# engagements 

In [6]:
sql_query = f"""
    SELECT
        week_commencing,
        page_id,
        CASE
            WHEN (AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user) 
                 > AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer)
            THEN ((AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user)) 
                 + (AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer))*0.04827
            ELSE (AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer)) 
                 + ((AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user))*0.04822
        END AS engaged_reach
    FROM 
        redshiftdb.central_insights.adverity_social_facebook_by_page AS p
    RIGHT JOIN
        world_service_audiences_insights.social_media_lookup_fb AS l
        ON p.page_id = l.fb_page_id
    WHERE 
        period = 'week'
    AND
        week_commencing BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['w/c_end']}'
    GROUP BY 
        week_commencing, page_id
        ;
"""

file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_engagements_redshift_extract.csv"
'''
df = execute_sql_query(sql_query)
df['page_id'] = df['page_id'].astype(str)
df.to_csv(file, index=False, na_rep='')
'''
try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = df['page_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")
    
facebook_engagements_raw = pd.read_csv(file, keep_default_na=False)
facebook_engagements_raw['page_id'] = platformID+facebook_engagements_raw['page_id'].astype(str)


no redshift connection using last time queried


In [9]:
facebook_engagements_raw['week_commencing'] = pd.to_datetime(facebook_engagements_raw['week_commencing'])
facebook_engagements_raw = facebook_engagements_raw.rename(columns={'page_id': 'Channel ID', 
                                                                    'week_commencing': 'w/c'})
print(facebook_engagements_raw.shape)

### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(facebook_engagements_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"6_{platformID}_engagements",
                                             "Check that all page IDs are found in SQL")

# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               id_column='Channel ID',
                                               main_data=facebook_engagements_raw,
                                               week_lookup=week_tester[['w/c']],
                                               test_number=f"7_{platformID}_engagements",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(facebook_engagements_raw, 
                           numeric_columns=['engaged_reach'], 
                           test_number=f"8_{platformID}_engagements",
                           test_step='Check no missing values in engaged_reach column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(facebook_engagements_raw, 
                               ['Channel ID', 'w/c'], 
                               test_number=f"9_{platformID}_engagements",
                               test_step='Check no duplicates from redshift returned')

(3754, 3)
...testing Channel ID...
❌ Test 6_FBE_engagements failed: not all elements from lookup were retrieved in dataset - decide if they are really missing or if these accounts are inactive 
...updating logbook...

❌ Test 7_FBE_engagements failed: Missing completed weeks detected.
...updating logbook...

✅ Test 8_FBE_engagements passed: No NaN and all numeric values > 0.
...updating logbook...

✅ Test 9_FBE_engagements passed: No combinations occurs more than once a week.
...updating logbook...



/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/test_functions.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  week_lookup[key] = pd.to_datetime(week_lookup[key])


## processing engagements

In [10]:
facebook_engagements = facebook_engagements_raw.merge(socialmedia_accounts[['Channel ID', 'ServiceID']], 
                                                      on='Channel ID', how='left', indicator=True)
test_functions.test_inner_join(facebook_engagements_raw, socialmedia_accounts, 
                               ['Channel ID'], 
                               f"10_{platformID}_engagements", 
                               test_step='checking social media accounts in lookup, adding service',
                               focus='left')


✅ Inner join test 10_FBE_engagements successful: No issues found.
...updating logbook...



In [11]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'ServiceID', 'w/c', 'engaged_reach']
facebook_engagements[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT.csv",
                                       index=None)

In [12]:
facebook_engagements[(facebook_engagements['w/c'] == '2025-12-01')  & 
    (facebook_engagements['Channel ID'].isin(['1296222730434431']))]

,w/c,Channel ID,engaged_reach,ServiceID,_merge


In [13]:
facebook_engagements

,w/c,Channel ID,engaged_reach,ServiceID,_merge
0,2025-06-16,FBE146214266618,1.356820e+03,WOR,both
1,2025-10-20,FBE1842714285954391,1.196975e+04,WOR,both
2,2025-11-24,FBE303522857815,1.463321e+06,POR,both
3,2025-04-14,FBE186742265162,1.250912e+06,TAM,both
4,2025-04-21,FBE186742265162,1.921171e+06,TAM,both
...,...,...,...,...,...
3749,2025-11-10,FBE173825495996416,2.294166e+04,WOR,both
3750,2025-11-10,FBE127031120644257,3.362021e+05,WOR,both
3751,2025-11-10,FBE510203778998227,2.955199e+02,MA-,both
3752,2025-06-02,FBE948946275170651,4.671522e+02,WSE,both


In [ ]:


#['1296222730434431' '143048895744759' '1526071940947174' '237647452933504''151955124848859']